# Notebook 05 — Anisotropy & Whitened CKA: Novel Technique 2

**TIN-7: Cross-Lingual Embedding Alignment Analysis**

Standard CKA can be confounded by **representation anisotropy** —
the phenomenon where all embedding vectors cluster in a narrow cone,
leading to artificially high similarity scores. This notebook
measures anisotropy, applies ZCA whitening, and compares standard
vs. whitened CKA to disentangle true cross-lingual alignment from
anisotropy artifacts.

## Research question

Is the cross-lingual similarity observed in Notebook 03 genuine,
or is it inflated by the anisotropic geometry of transformer
representations?

In [ ]:
import logging
import sys
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.languages import Language
from src.analysis.cross_lingual_embedding_alignment.cka import whitened_cka, linear_cka
from src.analysis.cross_lingual_embedding_alignment.visualization import (
    plot_convergence_curve,
    plot_anisotropy_heatmap,
    plot_eigenvalue_spectrum,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

RESULTS_DIR = PROJECT_ROOT / "analysis" / "results" / "cross_lingual"
FIGURES_DIR = RESULTS_DIR / "figures"
METRICS_DIR = RESULTS_DIR / "metrics"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Cached Activations

In [ ]:
# Load activations from disk.
languages = list(Language)
language_names = [lang.lang_name for lang in languages]
iso_names = [lang.iso_code for lang in languages]
n_layers = 4

activations: dict[str, dict[str, torch.Tensor]] = {}
for lang in languages:
    activations[lang.lang_name] = {}
    for layer_idx in range(n_layers):
        filepath = RESULTS_DIR / "activations" / f"layer_{layer_idx}_{lang.lang_name}.pt"
        if filepath.exists():
            activations[lang.lang_name][f"layer_{layer_idx}"] = torch.load(
                filepath, weights_only=True
            )

sample_shape = next(iter(next(iter(activations.values())).values())).shape
print(f"Loaded activations for {len(activations)} languages, {n_layers} layers.")
print(f"Activation shape: {sample_shape}")

## 2. Measure Representation Anisotropy

Anisotropy is measured as the average cosine similarity between
random pairs of sentence embeddings within each language at each
layer. Values close to 1.0 indicate high anisotropy (all vectors
point in similar directions).

Typical transformer models show anisotropy values of 0.5-0.9,
especially in later layers.

In [ ]:
def compute_anisotropy(embeddings: torch.Tensor, n_pairs: int = 5000) -> float:
    """Compute anisotropy as average cosine similarity of random pairs.

    Args:
        embeddings: Tensor of shape (n_sentences, hidden_dim).
        n_pairs: Number of random pairs to sample.

    Returns:
        Average cosine similarity (float in [-1, 1]).
    """
    n = embeddings.shape[0]
    if n < 2:
        return 0.0

    # Normalize to unit vectors.
    norms = embeddings.norm(dim=1, keepdim=True).clamp(min=1e-10)
    normalized = embeddings / norms

    # Sample random pairs.
    n_pairs = min(n_pairs, n * (n - 1) // 2)
    idx_a = torch.randint(0, n, (n_pairs,))
    idx_b = torch.randint(0, n, (n_pairs,))
    # Ensure different indices.
    mask = idx_a != idx_b
    idx_a, idx_b = idx_a[mask], idx_b[mask]

    # Cosine similarity = dot product of unit vectors.
    cos_sims = (normalized[idx_a] * normalized[idx_b]).sum(dim=1)
    return float(cos_sims.mean())


# Compute anisotropy for all languages and layers.
anisotropy_matrix = np.zeros((len(languages), n_layers))

for i, lang in enumerate(languages):
    for layer_idx in range(n_layers):
        layer_name = f"layer_{layer_idx}"
        emb = activations[lang.lang_name][layer_name]
        anisotropy_matrix[i, layer_idx] = compute_anisotropy(emb)

print("=== Anisotropy Scores ===")
print(f"{'Language':<12}", end="")
for l in range(n_layers):
    print(f"  Layer {l}", end="")
print()
for i, lang in enumerate(languages):
    print(f"{lang.iso_code:<12}", end="")
    for l in range(n_layers):
        print(f"  {anisotropy_matrix[i, l]:.4f}", end="")
    print()

## 3. Anisotropy Heatmap

In [ ]:
fig = plot_anisotropy_heatmap(
    anisotropy_scores=anisotropy_matrix,
    language_names=iso_names,
    layer_indices=list(range(n_layers)),
    save_path=str(FIGURES_DIR / "anisotropy_heatmap.png"),
)
plt.show()

## 4. Eigenvalue Spectrum Analysis

The eigenvalue spectrum of the representation covariance matrix
reveals the intrinsic dimensionality. A sharply decaying spectrum
means the representations live in a low-dimensional subspace
(high anisotropy).

In [ ]:
eigenvalues_by_layer: dict[int, np.ndarray] = {}

for layer_idx in range(n_layers):
    layer_name = f"layer_{layer_idx}"

    # Pool all languages together for the spectrum.
    all_emb = torch.cat(
        [activations[lang.lang_name][layer_name] for lang in languages],
        dim=0,
    )

    # Compute covariance eigenvalues.
    centered = all_emb - all_emb.mean(dim=0, keepdim=True)
    cov = (centered.T @ centered) / centered.shape[0]
    eigenvalues = torch.linalg.eigvalsh(cov)

    # Sort descending.
    eigenvalues = eigenvalues.flip(0).numpy()
    eigenvalues_by_layer[layer_idx] = eigenvalues

fig = plot_eigenvalue_spectrum(
    eigenvalues_by_layer=eigenvalues_by_layer,
    top_k=50,
    save_path=str(FIGURES_DIR / "eigenvalue_spectrum.png"),
)
plt.show()

## 5. Whitened CKA Matrices

Apply ZCA whitening before CKA computation to remove anisotropy
effects, then compare against standard linear CKA.

In [ ]:
# Compute whitened CKA matrices.
whitened_matrices: dict[int, np.ndarray] = {}
n_langs = len(languages)

for layer_idx in range(n_layers):
    layer_name = f"layer_{layer_idx}"
    matrix = np.zeros((n_langs, n_langs), dtype=np.float64)

    print(f"Computing whitened CKA for layer {layer_idx}...")
    for i in range(n_langs):
        matrix[i, i] = 1.0
        for j in range(i + 1, n_langs):
            X = activations[language_names[i]][layer_name]
            Y = activations[language_names[j]][layer_name]
            score = whitened_cka(X, Y).item()
            matrix[i, j] = score
            matrix[j, i] = score

    whitened_matrices[layer_idx] = matrix

# Save whitened CKA matrices.
cka_dir = RESULTS_DIR / "cka_matrices"
cka_dir.mkdir(parents=True, exist_ok=True)
for layer_idx, matrix in whitened_matrices.items():
    np.save(cka_dir / f"layer_{layer_idx}_whitened_cka.npy", matrix)

## 6. Standard vs. Whitened CKA Convergence Comparison

In [ ]:
# Load standard CKA matrices.
standard_matrices: dict[int, np.ndarray] = {}
for layer_idx in range(n_layers):
    filepath = RESULTS_DIR / "cka_matrices" / f"layer_{layer_idx}_linear_cka.npy"
    if filepath.exists():
        standard_matrices[layer_idx] = np.load(filepath)

# Compute average cross-lingual CKA for both variants.
def avg_off_diagonal(matrix: np.ndarray) -> float:
    """Compute mean of upper-triangle (off-diagonal) elements."""
    n = matrix.shape[0]
    return float(matrix[np.triu_indices(n, k=1)].mean())

layer_indices = list(range(n_layers))
standard_avg = [avg_off_diagonal(standard_matrices[l]) for l in layer_indices]
whitened_avg = [avg_off_diagonal(whitened_matrices[l]) for l in layer_indices]

fig = plot_convergence_curve(
    layer_indices=layer_indices,
    avg_cka_per_layer=standard_avg,
    whitened_cka_per_layer=whitened_avg,
    threshold=0.75,
    title="Standard vs. Whitened CKA Convergence",
    save_path=str(FIGURES_DIR / "convergence_standard_vs_whitened.png"),
)
plt.show()

## 7. Difference Heatmap: Whitened - Standard CKA

In [ ]:
import seaborn as sns

for layer_idx in layer_indices:
    diff = whitened_matrices[layer_idx] - standard_matrices[layer_idx]

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        diff,
        xticklabels=iso_names,
        yticklabels=iso_names,
        annot=True,
        fmt=".3f",
        cmap="RdBu_r",
        center=0,
        vmin=-0.5,
        vmax=0.5,
        square=True,
        linewidths=0.5,
        ax=ax,
    )
    ax.set_title(
        f"Whitened - Standard CKA (Layer {layer_idx})",
        fontsize=13,
        fontweight="bold",
    )
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / f"cka_diff_whitened_layer_{layer_idx}.png", bbox_inches="tight")
    plt.show()

## 8. Save Metrics

In [ ]:
anisotropy_metrics = {}
for i, lang in enumerate(languages):
    for layer_idx in range(n_layers):
        key = f"layer_{layer_idx}_{lang.lang_name}_anisotropy"
        anisotropy_metrics[key] = float(anisotropy_matrix[i, layer_idx])

for layer_idx in range(n_layers):
    anisotropy_metrics[f"layer_{layer_idx}_avg_whitened_cka"] = whitened_avg[layer_idx]
    anisotropy_metrics[f"layer_{layer_idx}_avg_standard_cka"] = standard_avg[layer_idx]

with open(METRICS_DIR / "anisotropy_metrics.json", "w") as f:
    json.dump(anisotropy_metrics, f, indent=2)

print(f"Metrics saved to {METRICS_DIR / 'anisotropy_metrics.json'}")

## 9. Summary

**Key findings:**
- **Extreme anisotropy across the board.** All 13 languages exhibit very high anisotropy
  (0.89–0.98), confirming that Tiny Aya's representations are heavily concentrated in a
  narrow cone. Anisotropy decreases gradually from layer 0 to layer 3 (e.g., English:
  0.977 → 0.943; Yoruba: 0.931 → 0.886), with Yoruba and Hindi consistently the least
  anisotropic and English the most.
- **Whitened CKA dramatically challenges the standard CKA picture.** Average cross-lingual
  standard CKA is moderate (0.65 → 0.64, layers 0–3), but whitened CKA jumps to near-perfect
  values (0.97 at layer 0, ≥0.999 at layers 1–3). This means that once anisotropy is removed,
  all language representations become almost identical.
- **The gap between standard and whitened CKA is very large** (roughly +0.32 at layer 0,
  +0.36 at layer 3), indicating that anisotropy *substantially* suppresses apparent
  cross-lingual similarity in standard CKA. The moderate standard CKA scores from
  Notebook 03 significantly understate the true alignment.
- **Interpretation:** The low standard CKA scores (∼0.64) are largely an artifact of
  anisotropic geometry, not genuine representational divergence. After whitening, Tiny Aya's
  representations are nearly language-agnostic from layer 1 onward, with all language pairs
  converging to CKA ≈ 1.0. This suggests the model has learned a shared multilingual
  representation space that is obscured by the narrow-cone geometry of transformer embeddings.